# Exercice 06 - Connexion a PostgreSQL

## Objectifs
- Se connecter a PostgreSQL depuis Spark
- Lire des tables SQL avec Spark
- Decouvrir la base Northwind
- Executer des requetes SQL sur les donnees

---

## 1. La base de donnees Northwind

**Northwind** est une base de donnees exemple classique qui simule une entreprise de vente.

```
+------------------------------------------------------------------+
|                    BASE NORTHWIND                                |
+------------------------------------------------------------------+
|                                                                  |
|  +------------+     +------------+     +------------+            |
|  | categories |     |  products  |     | suppliers  |            |
|  +------------+     +------------+     +------------+            |
|        |                  |                  |                   |
|        +--------+---------+------------------+                   |
|                 |                                                |
|                 v                                                |
|  +------------+     +------------+     +------------+            |
|  |   orders   |---->|order_details|<---|  products  |            |
|  +------------+     +------------+     +------------+            |
|        |                                                         |
|        v                                                         |
|  +------------+     +------------+                               |
|  | customers  |     | employees  |                               |
|  +------------+     +------------+                               |
|                                                                  |
+------------------------------------------------------------------+

Tables principales :
- customers    : clients de l'entreprise
- products     : catalogue de produits
- orders       : commandes passees
- order_details: details de chaque commande
- employees    : employes de l'entreprise
- categories   : categories de produits
- suppliers    : fournisseurs
```

## 2. Configuration de Spark pour PostgreSQL

In [ ]:
from pyspark.sql import SparkSession

# Creer la SparkSession
spark = SparkSession.builder \
    .appName("Spark PostgreSQL") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

print("Spark pret !")

In [ ]:
# Configuration de la connexion PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
connection_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

print("Configuration PostgreSQL :")
print(f"  URL: {jdbc_url}")
print(f"  User: {connection_properties['user']}")

## 3. Lire une table SQL

In [ ]:
# Lire la table customers
df_customers = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=connection_properties
)

print("Table customers chargee !")
print(f"Nombre de clients : {df_customers.count()}")

In [ ]:
# Afficher les premieres lignes
df_customers.show(5)

In [ ]:
# Afficher le schema
df_customers.printSchema()

## 4. Lire d'autres tables

In [ ]:
# Lire la table products
df_products = spark.read.jdbc(
    url=jdbc_url,
    table="products",
    properties=connection_properties
)

print(f"Nombre de produits : {df_products.count()}")
df_products.show(5)

In [ ]:
# Lire la table orders
df_orders = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_properties
)

print(f"Nombre de commandes : {df_orders.count()}")
df_orders.show(5)

In [ ]:
# Lire la table order_details
df_order_details = spark.read.jdbc(
    url=jdbc_url,
    table="order_details",
    properties=connection_properties
)

print(f"Nombre de lignes de commande : {df_order_details.count()}")
df_order_details.show(5)

## 5. Utiliser SQL avec Spark

Spark permet d'utiliser du SQL sur les DataFrames.

In [ ]:
# Enregistrer les DataFrames comme vues temporaires
df_customers.createOrReplaceTempView("customers")
df_products.createOrReplaceTempView("products")
df_orders.createOrReplaceTempView("orders")
df_order_details.createOrReplaceTempView("order_details")

print("Vues SQL creees !")

In [ ]:
# Requete SQL : clients par pays
result = spark.sql("""
    SELECT country, COUNT(*) as nb_clients
    FROM customers
    GROUP BY country
    ORDER BY nb_clients DESC
""")

result.show()

In [ ]:
# Requete SQL : produits les plus chers
result = spark.sql("""
    SELECT product_name, unit_price
    FROM products
    ORDER BY unit_price DESC
    LIMIT 10
""")

result.show()

In [ ]:
# Requete SQL : montant total des commandes par client
result = spark.sql("""
    SELECT 
        c.company_name,
        COUNT(DISTINCT o.order_id) as nb_commandes,
        ROUND(SUM(od.unit_price * od.quantity), 2) as montant_total
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_details od ON o.order_id = od.order_id
    GROUP BY c.company_name
    ORDER BY montant_total DESC
    LIMIT 10
""")

print("Top 10 clients par chiffre d'affaires :")
result.show(truncate=False)

## 6. Lire avec une requete SQL personnalisee

In [ ]:
# Au lieu de lire une table entiere, on peut executer une requete
query = "(SELECT customer_id, company_name, country FROM customers WHERE country = 'France') as french_customers"

df_french = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

print("Clients francais :")
df_french.show()

## 7. Fonction utilitaire pour lire les tables

In [ ]:
def lire_table(nom_table):
    """Fonction pour lire une table PostgreSQL"""
    return spark.read.jdbc(
        url="jdbc:postgresql://postgres:5432/app",
        table=nom_table,
        properties={
            "user": "postgres",
            "password": "postgres",
            "driver": "org.postgresql.Driver"
        }
    )

# Exemple d'utilisation
df_categories = lire_table("categories")
df_categories.show()

---

## Exercice

**Objectif** : Explorer la base Northwind avec Spark SQL

**Consigne** :
1. Lisez la table `employees`
2. Affichez le schema et les premieres lignes
3. Creez une vue temporaire
4. Ecrivez une requete SQL pour trouver :
   - Le nombre d'employes par ville
   - Les employes embauches apres 1993

A vous de jouer :

In [ ]:
# TODO: Lire la table employees
df_employees = lire_table("employees")

# TODO: Afficher le schema et les premieres lignes

In [ ]:
# TODO: Creer une vue temporaire

# TODO: Nombre d'employes par ville

In [ ]:
# TODO: Employes embauches apres 1993
# Indice : utilisez WHERE hire_date > '1993-12-31'

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **configurer Spark pour PostgreSQL**
- Comment **lire des tables** avec `spark.read.jdbc()`
- La structure de la base **Northwind**
- Comment utiliser **Spark SQL** pour analyser les donnees
- Comment lire avec une **requete personnalisee**

### Prochaine etape
Dans le prochain notebook, nous apprendrons a ingerer les donnees PostgreSQL vers notre Data Lake MinIO.